In [18]:
import numpy as np
import tensorflow as tf
print("The modules are imported successfully!")

The modules are imported successfully!


In [19]:
import numpy as np

class NeuralNetwork:
    '''
    layers should be a list where the number at index 0 should tell about the number of features your data has and further the number of neurons for each layer
    '''
    def __init__(self, layers):
        self.layers = len(layers) - 1
        # Tip: randn (normal distribution) is usually better than rand (uniform 0 to 1) 
        # for neural networks to avoid exploding/vanishing gradients early on.
        self.weights = [np.random.randn(layers[i], layers[i+1]) * 0.1 for i in range(len(layers) - 1)]
        self.activations = []
        self.z_values = []
        self.loss_history = [] # Renamed to avoid collision
        self.delta = None
    
    # Added 'self' parameter
    def relu(self, x):
        return np.maximum(0, x)
    
    # Added 'self' parameter
    def relu_derivative(self, x):
        return np.where(x > 0, 1, 0)
    
    # Renamed to avoid collision
    def calculate_loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def forward_pass(self, x):
        self.activations = [x]
        self.z_values = []
        for i in range(self.layers):
            z = np.dot(self.activations[-1], self.weights[i])
            self.z_values.append(z)
            if i < self.layers - 1:
                a = self.relu(z)
            else:
                a = z 
            self.activations.append(a)
        return self.activations[-1]
    
    def backward_pass(self, y_true, learning_rate):
        m = y_true.shape[0]
        self.delta = (2/m) * (self.activations[-1] - y_true)
        
        for i in reversed(range(self.layers)):
            current_activation = self.activations[i]
            grad_w = np.dot(current_activation.T, self.delta)
            
            if i > 0:
                prev_z = self.z_values[i-1]
                self.delta = np.dot(self.delta, self.weights[i].T) * self.relu_derivative(prev_z)
            
            self.weights[i] -= learning_rate * grad_w
        # NEW: Calculate and return the gradient w.r.t the INPUT of the dense network
        # This is the crucial package we must hand back to the CNN.
        grad_dense_input = np.dot(self.delta, self.weights[0].T)
        return grad_dense_input
        

    def train(self, X, y, epochs, learning_rate):
        self.loss_history = [] # Use renamed list
        for epoch in range(epochs):
            self.forward_pass(X)
            self.backward_pass(y, learning_rate)
            # Use renamed list and method
            current_loss = self.calculate_loss(y, self.activations[-1])
            self.loss_history.append(current_loss)
            
            # Optional: Print loss every 10 epochs to see it learning!
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {current_loss}")

In [20]:
class SimpleCNN(NeuralNetwork):
    def __init__(self, input_shape, kernel_sizes,dense_layers, strides=1, padding='valid'):
        """
        kernel_sizes: list like [3, 3, 5] for the f dimensions.
        dense_layers: list like [x, 128, 64] for the number of neurons in each dense layer after convolution. The first number (x) is just a placeholder and will be determined after the convolutional layers.
        """
        h,w = input_shape[-2], input_shape[-1]
        for i in range(len(kernel_sizes)):
            h,w = h - kernel_sizes[i] +1, w - kernel_sizes[i] +1
        dense_layers.insert(0, h*w) # Insert the flattened size after convolution as the first layer for the dense part
        
        super().__init__(dense_layers) # Initialize the dense layers using the parent class
        self.input_shape = input_shape
        self.kernel_sizes = kernel_sizes
        self.strides = strides
        self.padding = padding
        


        # Initialize kernels. 
        # Note: In a full CNN, kernels are 3D (Channels, Height, Width). 
        # We are keeping it 2D here for the simplest case.
        self.kernels = [
            np.random.normal(0, 1, size=(k, k)) for k in self.kernel_sizes
        ]
        
        self.c_activations = []
        self.c_delta = None

    def convolve(self, x, kernel):
        f_h, f_w = kernel.shape
        
        # 1. Apply Padding if requested
        if self.padding == 'same':
            # Formula: p = (f - 1) / 2
            pad_h = (f_h - 1) // 2
            pad_w = (f_w - 1) // 2
            # Pad with 0s on top/bottom and left/right
            x_padded = np.pad(x, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
        else:
            x_padded = x

        h_padded, w_padded = x_padded.shape

        # 2. Calculate correct Output Dimensions
        out_h = ((h_padded - f_h) // self.strides) + 1
        out_w = ((w_padded - f_w) // self.strides) + 1
        feature_map = np.zeros((out_h, out_w))

        # 3. Slide the Window
        for i in range(out_h):
            for j in range(out_w):
                # Calculate the exact starting pixels on the input
                start_i = i * self.strides
                start_j = j * self.strides
                
                # Slice the region
                region = x_padded[start_i : start_i + f_h, start_j : start_j + f_w]
                
                # Element-wise multiplication and sum
                feature_map[i, j] = np.sum(region * kernel)
                
        return feature_map

    def forward_pass_c(self, x):
        self.c_activations = []
        self.c_inputs = [] # NEW: We must track what goes INTO each layer
        
        current_input = x
        for kernel in self.kernels:
            self.c_inputs.append(current_input) # Save the input for backprop
            
            conv_out = self.convolve(current_input, kernel)
            current_input = np.maximum(0, conv_out) # ReLU Activation
            
            self.c_activations.append(current_input)
            
        return current_input

    # def calculate_loss_c(self, y_true, y_pred):
    #     return np.mean((y_true - y_pred) ** 2)
    

            
    def backward_pass_c(self, grad_from_dense, learning_rate):
        # 1. Un-flatten the incoming error from the dense network
        current_delta = grad_from_dense.reshape(self.c_activations[-1].shape)
        
        for i in reversed(range(len(self.kernels))):
            layer_input = self.c_inputs[i]
            kernel = self.kernels[i]
            
            # --- STEP A: Calculate Gradient of the Kernel (dL/dW) ---
            # Math: layer_input * current_delta
            grad_k = np.zeros_like(kernel)
            # (Your sliding window logic to calculate grad_k goes here, 
            # making sure to slide over 'layer_input', not activations[0])
            for m in range(current_delta.shape[0]):
                for n in range(current_delta.shape[1]):
        
            # 1. Find the 3x3 patch of the INPUT that created this output pixel
                    region = layer_input[m : m + kernel.shape[0], n : n + kernel.shape[1]]
            # 2. Multiply the input patch by this pixel's exact error, and add it
                    grad_k += region * current_delta[m, n]
            
            # Update the weights
            self.kernels[i] -= learning_rate * grad_k
            
            # STEP B: Calculate Gradient of the Input (next_delta)

            # 1. Flip the kernel 180 degrees
            flipped_kernel = np.rot90(kernel, 2) 

            # 2. Calculate full padding: p = filter_size - 1
            pad_h = kernel.shape[0] - 1
            pad_w = kernel.shape[1] - 1

            # 3. Add the zero-padding around our error grid
            padded_delta = np.pad(current_delta, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')

            # 4. Create an empty grid the exact size of the input image
            next_delta = np.zeros_like(layer_input)

            # 5. Slide the flipped kernel over the padded error grid
            for m in range(next_delta.shape[0]):
                for n in range(next_delta.shape[1]):

                    region = padded_delta[m : m + flipped_kernel.shape[0], n : n + flipped_kernel.shape[1]]
                    next_delta[m, n] = np.sum(region * flipped_kernel)

            # This becomes the error for the next layer down the chain!
            current_delta = next_delta

    def train_cnn(self, X_train, y_train, epochs, learning_rate):
        self.loss_history = []
        for epoch in range(epochs):
            total_loss = 0
            
            # NEW: Loop through each image one by one
            for idx in range(len(X_train)):
                X = X_train[idx]
                y = np.array([[y_train[idx]]]) # Ensure y is shape (1,1) for the dense layer
                
                # --- FORWARD PASS ---
                dunk_feature_map = self.forward_pass_c(X)
                flattened_input = dunk_feature_map.flatten().reshape(1, -1)
                pred = self.forward_pass(flattened_input)
                
                # --- BACKWARD PASS ---
                grad_at_flatten_layer = self.backward_pass(y, learning_rate)
                self.backward_pass_c(grad_at_flatten_layer, learning_rate)
                
                total_loss += self.calculate_loss(y, pred)

                
            # Record average loss for the epoch
            self.loss_history.append(total_loss / len(X_train))
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Average Loss: {self.loss_history[-1]:.4f}")


    


In [21]:
shield_dunk_1 = np.array([
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1]
])

shield_dunk_2 = np.array([ # Slightly shifted star
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0]
])

# Prince Aerion's Shield (Fiery block) - Class 0
shield_aerion_1 = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
])

shield_aerion_2 = np.array([ # Smaller fire
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0]
])

# Prepare Dataset
X_train = [shield_dunk_1, shield_aerion_1, shield_dunk_2, shield_aerion_2]
y_train = [1, 0, 1, 0] # 1 for Dunk, 0 for Aerion

# Initialize Model
# 1 Conv Layer with a 3x3 filter, followed by a Dense Network: Flatten -> 10 neurons -> 1 output neuron
model = SimpleCNN(input_shape=(5, 5), kernel_sizes=[3], dense_layers=[10, 1])

# Train!
model.train_cnn(X_train, y_train, epochs=100, learning_rate=0.05)
print("\n--- Final Tourney Verdict ---")
dunk_pred = model.forward_pass(model.forward_pass_c(shield_dunk_1).flatten().reshape(1,-1))
aerion_pred = model.forward_pass(model.forward_pass_c(shield_aerion_1).flatten().reshape(1,-1))

print(f"Ser Duncan Shield Prediction (Expected ~1.0): {dunk_pred[0][0]:.4f}")
print(f"Prince Aerion Shield Prediction (Expected ~0.0): {aerion_pred[0][0]:.4f}")

Epoch 10, Average Loss: 0.1923
Epoch 20, Average Loss: 0.0024
Epoch 30, Average Loss: 0.0001
Epoch 40, Average Loss: 0.0000
Epoch 50, Average Loss: 0.0000
Epoch 60, Average Loss: 0.0000
Epoch 70, Average Loss: 0.0000
Epoch 80, Average Loss: 0.0000
Epoch 90, Average Loss: 0.0000
Epoch 100, Average Loss: 0.0000

--- Final Tourney Verdict ---
Ser Duncan Shield Prediction (Expected ~1.0): 1.0000
Prince Aerion Shield Prediction (Expected ~0.0): 0.0000
